# 📊 04 — Análisis y Visualización del Mercado Laboral Tech en España
**Cliente:** DataTalent Solutions S.L.  
**Versión:** 3.0 · Datos reales desde `02_cleaning.ipynb` y `03_eda.ipynb`  
**Librerías:** `pandas` · `numpy` · `matplotlib` · `seaborn` · `plotly` · `squarify` · `scipy` · `statsmodels`

---

### Estructura del análisis

| Bloque | Contenido |
|--------|-----------|
| 0 | Configuración, carga de datos reales y calidad |
| 1 | Distribución y volumen de vacantes |
| 2 | Análisis retributivo y salarial |
| 3 | Mapas de calor y correlaciones (incluye heatmap de co-ocurrencia de skills) |
| 4 | Stack tecnológico y brecha Used vs Wanted |
| 5 | Análisis estadístico avanzado |
| 6 | Visualizaciones interactivas (Plotly) |
| 7 | Conclusiones y exportación |

> **Requisitos:** Ejecutar primero `02_cleaning.ipynb` y `03_eda.ipynb`. Este notebook carga los CSVs limpios desde `data/clean/` y los datos post-EDA desde `data/eda/`. No se generan datos sintéticos.


## 🛠️ Bloque 0 — Configuración e Importaciones

In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS (Google Colab)
# ============================================================
import subprocess, sys

def install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

install_if_missing('squarify')
install_if_missing('statsmodels')
install_if_missing('plotly')
print("✅ Dependencias verificadas.")


In [ ]:
# ============================================================
# CONFIGURACION E IMPORTACIONES
# ============================================================
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from IPython.display import display

try:
    import squarify
    HAS_SQUARIFY = True
except ImportError:
    squarify = None
    HAS_SQUARIFY = False

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    sm = None
    HAS_STATSMODELS = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


def find_project_root(start=None):
    """Localiza la raíz del proyecto."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'data' / 'clean').exists():
            return candidate
    return current.parent if current.name.lower() == 'notebooks' else current


PROJECT_ROOT = find_project_root()
DATA_CLEAN_DIR = PROJECT_ROOT / 'data' / 'clean'
DATA_EDA_DIR   = PROJECT_ROOT / 'data' / 'eda'
IMAGES_DIR     = PROJECT_ROOT / 'images'
IMAGES_DIR.mkdir(exist_ok=True)

# ============================================================
# PALETA Y ESTILO CORPORATIVO
# ============================================================
PALETA = {
    'primary':   '#1A365D',
    'secondary': '#2B6CB0',
    'accent':    '#4299E1',
    'warm':      '#ED8936',
    'success':   '#48BB78',
    'muted':     '#A0AEC0',
    'dark':      '#171923',
    'light':     '#EBF8FF',
}
PALETTE_LIST = list(PALETA.values())

sns.set_theme(style='whitegrid', palette=PALETTE_LIST)
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'figure.titlesize': 18,
    'figure.titleweight': 'bold',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': '#FAFAFA',
})

print('✅ Entorno configurado correctamente.')
print(f'📁 Raíz del proyecto: {PROJECT_ROOT}')
print(f'📂 Datos post-EDA: {DATA_EDA_DIR}')
print(f'📂 Datos limpios: {DATA_CLEAN_DIR}')
print(f'🖼️  Gráficos: {IMAGES_DIR}')
if not HAS_SQUARIFY:
    print('⚠️  squarify no instalado: treemap se sustituye por barras.')
if not HAS_STATSMODELS:
    print('⚠️  statsmodels no instalado: se usará scipy.')


### 📥 Carga de datos reales desde el pipeline

Este notebook **requiere** los archivos generados por `02_cleaning.ipynb` y `03_eda.ipynb`.  
No se generan datos sintéticos: si faltan los CSVs, el notebook lanza un error descriptivo.

**Archivos esperados (en orden de preferencia):**
- `data/eda/jobs_eda.csv` → salida de `03_eda.ipynb` (preferido)
- `data/clean/jobs_all_clean.csv` → salida de `02_cleaning.ipynb` (fallback)
- `data/clean/technology_rankings_used.csv`
- `data/clean/technology_rankings_wanted.csv`
- `data/clean/job_skills_long.csv`


In [ ]:
# ============================================================
# CARGA DE DATOS REALES (sin datos sintéticos)
# ============================================================

def read_first_available(candidates, label):
    """Lee el primer archivo que exista de la lista de candidatos."""
    for path in candidates:
        if path.exists():
            df_tmp = pd.read_csv(path)
            print(f"✅ {label} cargado desde '{path.relative_to(PROJECT_ROOT)}' → {df_tmp.shape}")
            return df_tmp, path
    paths_str = [str(p.relative_to(PROJECT_ROOT)) for p in candidates]
    raise FileNotFoundError(
        f"No se encontró '{label}'. Archivos esperados:\n  " + "\n  ".join(paths_str) +
        "\n\n👉 Ejecuta primero 02_cleaning.ipynb (y opcionalmente 03_eda.ipynb)."
    )

# Dataset principal de ofertas
df, JOBS_SOURCE_PATH = read_first_available(
    [DATA_EDA_DIR / 'jobs_eda.csv', DATA_CLEAN_DIR / 'jobs_all_clean.csv'],
    'Dataset principal de ofertas'
)
DATOS_POST_EDA = (JOBS_SOURCE_PATH.parent == DATA_EDA_DIR)

# Rankings tecnológicos
df_used, USED_SOURCE_PATH = read_first_available(
    [DATA_EDA_DIR / 'technology_rankings_used_eda.csv', DATA_CLEAN_DIR / 'technology_rankings_used.csv'],
    'Ranking tecnologías usadas'
)
df_wanted, WANTED_SOURCE_PATH = read_first_available(
    [DATA_EDA_DIR / 'technology_rankings_wanted_eda.csv', DATA_CLEAN_DIR / 'technology_rankings_wanted.csv'],
    'Ranking tecnologías demandadas'
)

# Dataset de skills en formato largo (para heatmap de co-ocurrencia)
try:
    df_skills_long, _ = read_first_available(
        [DATA_EDA_DIR / 'job_skills_long_eda.csv', DATA_CLEAN_DIR / 'job_skills_long.csv'],
        'Skills en formato largo'
    )
    HAS_SKILLS_LONG = True
except FileNotFoundError as exc:
    print(f"⚠️  {exc}")
    print("   El heatmap de co-ocurrencia de skills no estará disponible.")
    df_skills_long = None
    HAS_SKILLS_LONG = False

# ── Columnas de skills binarias disponibles en df ──
SKILL_CANDIDATES = ['Python','SQL','Power BI','AWS','Excel','Tableau','Spark',
                    'Databricks','Docker','Git','R','TensorFlow','PyTorch','Azure','GCP']
SKILLS_COLS = [s for s in SKILL_CANDIDATES if s in df.columns]

# Fuente de información
if DATOS_POST_EDA:
    print("\n📌 Usando datos post-EDA (03_eda.ipynb).")
else:
    print("\n📌 Usando datos del cleaning (02_cleaning.ipynb) como fuente principal.")
    print("   Ejecuta 03_eda.ipynb para activar enriquecimientos adicionales.")

print(f"\n📋 Columnas disponibles: {df.columns.tolist()}")
print(f"\n📊 Resumen estadístico básico:")
display(df.describe(include='all').T)


### 🔍 Calidad del dataset

In [ ]:
# ============================================================
# DASHBOARD DE CALIDAD DE DATOS
# ============================================================
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
tipos = df.dtypes

calidad = pd.DataFrame({
    'Tipo': tipos,
    'Nulos': nulos,
    'Nulos (%)': nulos_pct,
    'Únicos': df.nunique(),
}).sort_values('Nulos (%)', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dashboard de Calidad del Dataset', fontsize=16, fontweight='bold', y=1.01)

# Mapa de calor de nulos
if nulos.sum() > 0:
    missing_matrix = df.isnull().astype(int)
    sns.heatmap(missing_matrix.T, cmap='YlOrRd', cbar=False, ax=axes[0],
                yticklabels=True, xticklabels=False)
    axes[0].set_title('Mapa de Valores Nulos')
    axes[0].set_xlabel('Registros')
else:
    axes[0].text(0.5, 0.5, '✅ Sin valores nulos', ha='center', va='center',
                 fontsize=16, transform=axes[0].transAxes)
    axes[0].set_title('Mapa de Valores Nulos')
    axes[0].axis('off')

# Barras de cobertura
cobertura = (1 - nulos_pct / 100).sort_values()
cols_show = cobertura[cobertura < 1.0].tail(15) if (cobertura < 1.0).any() else cobertura.tail(10)
colors_cob = ['#48BB78' if v >= 0.9 else '#ED8936' if v >= 0.7 else '#FC8181' for v in cols_show.values]
axes[1].barh(cols_show.index, cols_show.values * 100, color=colors_cob)
axes[1].set_title('Cobertura por Columna (%)')
axes[1].set_xlabel('Cobertura (%)')
axes[1].set_xlim(0, 105)
for bar, val in zip(axes[1].patches, cols_show.values * 100):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '00_calidad_datos.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"\n📊 Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"   Columnas con nulos: {(nulos > 0).sum()}")
display(calidad[calidad['Nulos'] > 0].head(15))


## 📍 Bloque 1 — Distribución y Volumen de Vacantes

Análisis de dónde provienen las ofertas, bajo qué modalidades se publican y qué niveles de seniority son más demandados.


In [ ]:
# ============================================================
# BLOQUE 1: DISTRIBUCIÓN Y VOLUMEN (Gráficos 1-3)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Distribución y Volumen de Vacantes', fontsize=18, fontweight='bold', y=1.02)

# Detectar columna de ciudad
if 'city_clean' in df.columns:
    city_col = 'city_clean'
elif 'city' in df.columns:
    city_col = 'city'
else:
    city_col = None

# G1: Ofertas por ciudad
if city_col:
    city_counts = df[city_col].value_counts().head(8)
    bars = axes[0].barh(city_counts.index, city_counts.values,
                        color=sns.color_palette('Blues_r', len(city_counts)))
    axes[0].set_title('1. Ofertas por Ciudad', pad=12)
    axes[0].set_xlabel('Nº Ofertas')
    for bar, val in zip(bars, city_counts.values):
        axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     str(val), va='center', fontsize=9)
else:
    axes[0].text(0.5, 0.5, 'Ciudad no disponible', ha='center', va='center',
                 transform=axes[0].transAxes, fontsize=12)
    axes[0].axis('off')

# G2: Modalidad de trabajo
modality_col = 'remote_modality' if 'remote_modality' in df.columns else ('is_remote' if 'is_remote' in df.columns else None)
if modality_col == 'is_remote':
    mod_map = {True: 'Remoto', False: 'Presencial/Híbrido'}
    mod_counts = df['is_remote'].map(mod_map).value_counts()
elif modality_col:
    mod_counts = df[modality_col].value_counts().head(6)
else:
    mod_counts = None

if mod_counts is not None:
    axes[1].pie(mod_counts.values, labels=mod_counts.index,
                autopct='%1.1f%%', colors=sns.color_palette('Blues_r', len(mod_counts)),
                startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    axes[1].set_title('2. Modalidad de Trabajo', pad=12)
else:
    axes[1].text(0.5, 0.5, 'Modalidad no disponible', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=12)
    axes[1].axis('off')

# G3: Seniority
sen_col = 'seniority_level' if 'seniority_level' in df.columns else ('seniority' if 'seniority' in df.columns else None)
if sen_col:
    sen_counts = df[sen_col].value_counts()
    bar_colors = sns.color_palette('Blues_r', len(sen_counts))
    axes[2].bar(sen_counts.index, sen_counts.values, color=bar_colors, edgecolor='white')
    axes[2].set_title('3. Distribución por Seniority', pad=12)
    axes[2].set_ylabel('Nº Ofertas')
    for bar, val in zip(axes[2].patches, sen_counts.values):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     str(val), ha='center', va='bottom', fontsize=9)
else:
    axes[2].text(0.5, 0.5, 'Seniority no disponible', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=12)
    axes[2].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_distribucion_volumen.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Gráficos 1-3 generados.")


## 💶 Bloque 2 — Análisis Retributivo y Salarial

Distribuciones salariales cruzadas con seniority, modalidad, ciudad y rol.


In [ ]:
# ============================================================
# BLOQUE 2: ANÁLISIS SALARIAL (Gráficos 4-8)
# ============================================================

sal_col = 'salary_clean' if 'salary_clean' in df.columns else 'salary'
outlier_col = 'salary_clean_outlier' if 'salary_clean_outlier' in df.columns else None

if outlier_col:
    df_sal = df[df[sal_col].notna() & ~df[outlier_col]].copy()
else:
    df_sal = df[df[sal_col].notna()].copy()

print(f"💶 Registros con salario válido (sin outliers): {len(df_sal):,}")
print(f"   Mediana: {df_sal[sal_col].median():,.0f} € | Media: {df_sal[sal_col].mean():,.0f} €")
print(f"   P10: {df_sal[sal_col].quantile(0.1):,.0f} € | P90: {df_sal[sal_col].quantile(0.9):,.0f} €")

sen_col = 'seniority_level' if 'seniority_level' in df_sal.columns else ('seniority' if 'seniority' in df_sal.columns else None)
modality_col = 'remote_modality' if 'remote_modality' in df_sal.columns else None
city_col = 'city_clean' if 'city_clean' in df_sal.columns else ('city' if 'city' in df_sal.columns else None)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis Salarial del Mercado Tech en España', fontsize=18, fontweight='bold')

# G4: Boxplot seniority
if sen_col and df_sal[sen_col].notna().any():
    order_sen = df_sal.groupby(sen_col)[sal_col].median().sort_values().index.tolist()
    sns.boxplot(x=sen_col, y=sal_col, data=df_sal, order=order_sen,
                palette='Blues_r', width=0.6, ax=axes[0,0])
    axes[0,0].set_title('4. Distribución Salarial por Seniority', pad=12)
    axes[0,0].set_xlabel('')
    axes[0,0].set_ylabel('Salario (€)')
    axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f} €'))
else:
    axes[0,0].text(0.5, 0.5, 'Seniority no disponible', ha='center', va='center',
                   transform=axes[0,0].transAxes, fontsize=12)
    axes[0,0].axis('off')

# G5: Histograma de salarios
if df_sal[sal_col].notna().any():
    axes[0,1].hist(df_sal[sal_col].dropna(), bins=35,
                   color=PALETA['secondary'], edgecolor='white', alpha=0.85)
    axes[0,1].axvline(df_sal[sal_col].median(), color=PALETA['warm'],
                      linestyle='--', linewidth=2, label=f"Mediana {df_sal[sal_col].median():,.0f} €")
    axes[0,1].axvline(df_sal[sal_col].mean(), color=PALETA['success'],
                      linestyle=':', linewidth=2, label=f"Media {df_sal[sal_col].mean():,.0f} €")
    axes[0,1].set_title('5. Distribución de Salarios', pad=12)
    axes[0,1].set_xlabel('Salario Bruto Anual (€)')
    axes[0,1].set_ylabel('Frecuencia')
    axes[0,1].legend()

# G6: Salario por modalidad
if modality_col and df_sal[modality_col].notna().any():
    order_mod = df_sal.groupby(modality_col)[sal_col].median().sort_values(ascending=False).index
    sns.violinplot(x=modality_col, y=sal_col, data=df_sal, order=order_mod,
                   palette='Blues_r', inner='quartile', ax=axes[1,0])
    axes[1,0].set_title('6. Salario por Modalidad', pad=12)
    axes[1,0].set_xlabel('')
    axes[1,0].set_ylabel('Salario (€)')
else:
    axes[1,0].text(0.5, 0.5, 'Modalidad no disponible', ha='center', va='center',
                   transform=axes[1,0].transAxes, fontsize=12)
    axes[1,0].axis('off')

# G7: Salario por ciudad (top 6)
if city_col and df_sal[city_col].notna().any():
    top_cities = df_sal[city_col].value_counts().head(6).index
    df_city = df_sal[df_sal[city_col].isin(top_cities)]
    order_city = df_city.groupby(city_col)[sal_col].median().sort_values(ascending=False).index
    sns.boxplot(x=city_col, y=sal_col, data=df_city, order=order_city,
                palette='Blues_r', width=0.5, ax=axes[1,1])
    axes[1,1].set_title('7. Salario por Ciudad (Top 6)', pad=12)
    axes[1,1].set_xlabel('')
    axes[1,1].set_ylabel('Salario (€)')
    axes[1,1].tick_params(axis='x', rotation=20)
else:
    axes[1,1].text(0.5, 0.5, 'Ciudad no disponible', ha='center', va='center',
                   transform=axes[1,1].transAxes, fontsize=12)
    axes[1,1].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_analisis_salarial.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Gráficos 4-7 generados.")

# G8: Salario por Rol
if 'job_title' in df_sal.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    order_roles = df_sal.groupby('job_title')[sal_col].median().sort_values().index
    sns.boxplot(x=sal_col, y='job_title', data=df_sal, order=order_roles,
                palette='Blues_r', width=0.5, fliersize=0, boxprops=dict(alpha=0.8), ax=ax)
    sns.stripplot(x=sal_col, y='job_title',
                  data=df_sal.sample(min(300, len(df_sal)), random_state=42),
                  order=order_roles, color=PALETA['warm'], alpha=0.3,
                  jitter=True, size=3, ax=ax)
    ax.set_title('8. Distribución Salarial por Rol', fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel('Salario Bruto Anual (€)')
    ax.set_ylabel('')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f} €'))
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02b_salario_por_rol.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✅ Gráfico 8 generado.")


## 🔥 Bloque 3 — Mapas de Calor y Análisis de Correlaciones

Los mapas de calor permiten identificar relaciones entre variables categóricas y numéricas.  
Incluye además un **heatmap de co-ocurrencia de skills** calculado a partir de `job_skills_long.csv`.


In [ ]:
# ============================================================
# BLOQUE 3: HEATMAPS (Gráficos 9-12)
# ============================================================

custom_cmap = LinearSegmentedColormap.from_list(
    'dt', [PALETA['light'], PALETA['secondary'], PALETA['primary']]
)

sen_col = 'seniority_level' if 'seniority_level' in df_sal.columns else ('seniority' if 'seniority' in df_sal.columns else None)
city_col = 'city_clean' if 'city_clean' in df_sal.columns else ('city' if 'city' in df_sal.columns else None)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Mapas de Calor y Correlaciones', fontsize=18, fontweight='bold')

# G9: Heatmap Seniority × Ciudad → Salario Medio
if sen_col and city_col and df_sal[sen_col].notna().any():
    top_cities_h = df_sal[city_col].value_counts().head(6).index
    pivot_sal = df_sal[df_sal[city_col].isin(top_cities_h)].pivot_table(
        values=sal_col, index=sen_col, columns=city_col, aggfunc='median'
    )
    sns.heatmap(pivot_sal, annot=True, fmt='.0f', cmap=custom_cmap,
                linewidths=0.5, linecolor='white', ax=axes[0,0],
                annot_kws={'fontsize': 10, 'fontweight': 'bold'},
                cbar_kws={'label': 'Salario Mediano (€)', 'shrink': 0.8})
    axes[0,0].set_title('9. Salario Mediano: Seniority × Ciudad', pad=12)
    axes[0,0].set_xlabel('')
    axes[0,0].set_ylabel('')
else:
    axes[0,0].text(0.5, 0.5, 'Datos insuficientes', ha='center', va='center',
                   transform=axes[0,0].transAxes, fontsize=12)
    axes[0,0].axis('off')

# G10: Heatmap Rol × Modalidad → Número de Ofertas
modality_col = 'remote_modality' if 'remote_modality' in df.columns else None
if 'job_title' in df.columns and modality_col:
    pivot_count = df.pivot_table(
        values='job_id' if 'job_id' in df.columns else df.columns[0],
        index='job_title', columns=modality_col, aggfunc='count'
    ).fillna(0).astype(int)
    sns.heatmap(pivot_count, annot=True, fmt='d', cmap='Blues',
                linewidths=0.5, linecolor='white', ax=axes[0,1],
                annot_kws={'fontsize': 10})
    axes[0,1].set_title('10. Nº Ofertas: Rol × Modalidad', pad=12)
    axes[0,1].set_xlabel('')
    axes[0,1].set_ylabel('')
else:
    axes[0,1].text(0.5, 0.5, 'Modalidad no disponible', ha='center', va='center',
                   transform=axes[0,1].transAxes, fontsize=12)
    axes[0,1].axis('off')

# G11: Heatmap correlación de variables numéricas
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols_clean = [c for c in num_cols if df[c].notna().sum() > len(df) * 0.3
                  and 'outlier' not in c.lower()]
if len(num_cols_clean) >= 2:
    corr_matrix = df[num_cols_clean].corr()
    mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask_upper, annot=True, fmt='.2f',
                cmap='RdYlBu_r', center=0,
                linewidths=0.5, linecolor='white', ax=axes[1,0],
                annot_kws={'fontsize': 9},
                cbar_kws={'shrink': 0.8})
    axes[1,0].set_title('11. Correlación entre Variables Numéricas', pad=12)
else:
    axes[1,0].text(0.5, 0.5, 'Variables numéricas insuficientes', ha='center', va='center',
                   transform=axes[1,0].transAxes, fontsize=12)
    axes[1,0].axis('off')

# G12: Heatmap skills binarias (si existen en df)
if SKILLS_COLS and len(SKILLS_COLS) >= 3:
    skills_corr = df[SKILLS_COLS].corr()
    sns.heatmap(skills_corr, annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.5, linecolor='white', ax=axes[1,1],
                annot_kws={'fontsize': 8},
                cbar_kws={'shrink': 0.8, 'label': 'Correlación'})
    axes[1,1].set_title('12. Correlación entre Skills (columnas binarias)', pad=12)
else:
    axes[1,1].text(0.5, 0.5, 'Skills binarias no disponibles en este dataset',
                   ha='center', va='center',
                   transform=axes[1,1].transAxes, fontsize=11,
                   wrap=True)
    axes[1,1].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '03_heatmaps.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Gráficos 9-12 generados.")


In [ ]:
# ── G13: Heatmap Sector × Seniority → Salario ──
sector_col = 'sector' if 'sector' in df_sal.columns else ('industry' if 'industry' in df_sal.columns else None)
sen_col = 'seniority_level' if 'seniority_level' in df_sal.columns else ('seniority' if 'seniority' in df_sal.columns else None)

if sector_col and sen_col:
    pivot_sec = df_sal.pivot_table(values=sal_col, index=sector_col,
                                   columns=sen_col, aggfunc='median')
    fig, ax = plt.subplots(figsize=(11, 7))
    custom_cmap2 = LinearSegmentedColormap.from_list('dt2',
        ['#EBF8FF', PALETA['accent'], PALETA['secondary'], PALETA['primary']])
    sns.heatmap(pivot_sec, annot=True, fmt='.0f', cmap=custom_cmap2,
                linewidths=0.8, linecolor='white', ax=ax,
                annot_kws={'fontsize': 11, 'fontweight': 'bold'},
                cbar_kws={'label': 'Salario Mediano (€)', 'shrink': 0.85})
    ax.set_title('13. Salario Mediano por Sector y Nivel (€)', fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '03b_heatmap_sector_seniority.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✅ Heatmap sector × seniority generado.")
else:
    print(f"ℹ️  Columnas sector/industry o seniority no encontradas. Disponibles: {df_sal.columns.tolist()}")


### 🆕 Gráfica de Calor — Co-ocurrencia de Skills en Ofertas

Esta nueva visualización muestra qué pares de skills aparecen juntos con más frecuencia en las ofertas de empleo,
usando el archivo `job_skills_long.csv` generado por `02_cleaning.ipynb`.


In [ ]:
# ============================================================
# NUEVA GRÁFICA: HEATMAP DE CO-OCURRENCIA DE SKILLS
# ============================================================

if HAS_SKILLS_LONG and df_skills_long is not None:
    # Pivotear a formato ancho: job_id × skill (binario)
    skill_col = 'skill'
    job_id_col = 'job_id'
    
    if skill_col in df_skills_long.columns and job_id_col in df_skills_long.columns:
        # Top N skills más frecuentes
        TOP_N_SKILLS = 15
        top_skills = df_skills_long[skill_col].value_counts().head(TOP_N_SKILLS).index.tolist()
        
        # Filtrar y crear tabla pivote
        df_sk_filtered = df_skills_long[df_skills_long[skill_col].isin(top_skills)].copy()
        df_sk_filtered['present'] = 1
        skill_pivot = df_sk_filtered.pivot_table(
            index=job_id_col, columns=skill_col, values='present', fill_value=0
        )
        skill_pivot = skill_pivot[top_skills]  # Ordenar por frecuencia
        
        # Calcular matriz de co-ocurrencia (dot product)
        co_matrix = skill_pivot.T.dot(skill_pivot)
        total_jobs = len(skill_pivot)
        
        # Normalizar: porcentaje de ofertas donde ambas skills co-ocurren
        co_matrix_pct = (co_matrix / total_jobs * 100).round(1)
        
        # Poner diagonal a 0 para que no domine visualmente
        np.fill_diagonal(co_matrix_pct.values, 0)
        
        # Ordenar por frecuencia total
        freq_order = df_sk_filtered[skill_col].value_counts().reindex(top_skills).sort_values(ascending=False).index
        co_matrix_pct = co_matrix_pct.reindex(index=freq_order, columns=freq_order)
        
        fig, ax = plt.subplots(figsize=(13, 10))
        
        cmap_cooc = LinearSegmentedColormap.from_list(
            'cooc', ['#F7FAFC', '#BEE3F8', PALETA['secondary'], PALETA['primary']]
        )
        
        # Máscara triángulo superior
        mask_upper = np.triu(np.ones_like(co_matrix_pct, dtype=bool), k=1)
        
        sns.heatmap(
            co_matrix_pct,
            mask=mask_upper,
            annot=True,
            fmt='.1f',
            cmap=cmap_cooc,
            linewidths=0.5,
            linecolor='#E2E8F0',
            ax=ax,
            annot_kws={'fontsize': 8.5, 'fontweight': 'bold'},
            cbar_kws={'label': '% Ofertas con ambas skills', 'shrink': 0.85},
            vmin=0,
        )
        
        ax.set_title(
            f'Co-ocurrencia de Skills en Ofertas (Top {TOP_N_SKILLS} skills)\n'
            '% de ofertas donde aparecen juntas las dos skills',
            fontsize=14, fontweight='bold', pad=18
        )
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.tick_params(axis='y', rotation=0, labelsize=10)
        
        # Añadir nota de la diagonal
        ax.text(0.98, -0.02, '(diagonal excluida — frecuencia individual)',
                transform=ax.transAxes, fontsize=8, color='gray', ha='right', va='top')
        
        plt.tight_layout()
        plt.savefig(IMAGES_DIR / '03c_heatmap_skills_coocurrencia.png', dpi=200, bbox_inches='tight')
        plt.show()
        
        print(f"✅ Heatmap de co-ocurrencia generado ({total_jobs:,} ofertas, Top {TOP_N_SKILLS} skills).")
        
        # Mini-tabla con los pares más frecuentes
        co_long = co_matrix_pct.where(mask_upper == False).stack().reset_index()
        co_long.columns = ['Skill A', 'Skill B', 'Co-ocurrencia (%)']
        co_long = co_long[co_long['Skill A'] != co_long['Skill B']].sort_values(
            'Co-ocurrencia (%)', ascending=False
        )
        print("\n🔝 Top 10 pares de skills que más co-ocurren:")
        display(co_long.head(10).reset_index(drop=True))
    else:
        print(f"⚠️  job_skills_long no contiene las columnas esperadas.")
        print(f"   Columnas disponibles: {df_skills_long.columns.tolist()}")
else:
    print("ℹ️  job_skills_long.csv no disponible.")
    print("   Ejecuta 02_cleaning.ipynb para generar este archivo.")
    print("\n   Alternativa: se mostrará la correlación de skills binarias (si existen en el dataset principal):")
    if SKILLS_COLS and len(SKILLS_COLS) >= 5:
        co_alt = df[SKILLS_COLS].T.dot(df[SKILLS_COLS])
        total = len(df)
        co_alt_pct = (co_alt / total * 100).round(1)
        np.fill_diagonal(co_alt_pct.values, 0)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(co_alt_pct, annot=True, fmt='.1f', cmap='Blues',
                    linewidths=0.5, linecolor='white', ax=ax,
                    annot_kws={'fontsize': 9},
                    cbar_kws={'label': '% Co-ocurrencia'})
        ax.set_title('Co-ocurrencia de Skills (columnas binarias del dataset)',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(IMAGES_DIR / '03c_heatmap_skills_coocurrencia_alt.png', dpi=200, bbox_inches='tight')
        plt.show()


## 🚀 Bloque 4 — Stack Tecnológico y Brecha Used vs Wanted

Análisis de los lenguajes y herramientas más demandados, comparando el uso real con la intención de aprendizaje.


In [ ]:
# ============================================================
# BLOQUE 4: STACK TECNOLÓGICO (Gráficos 14-17)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Stack Tecnológico y Demanda', fontsize=18, fontweight='bold')

# G14: Top skills en ofertas (de df principal)
if SKILLS_COLS:
    pct_skills = (df[SKILLS_COLS].sum() / len(df) * 100).sort_values(ascending=False)
    colors_sk = sns.color_palette('Blues_r', len(pct_skills))
    bars_sk = axes[0,0].barh(pct_skills.index, pct_skills.values, color=colors_sk, edgecolor='white')
    axes[0,0].set_title('14. Penetración de Skills en Ofertas (%)', pad=12)
    axes[0,0].set_xlabel('% de Ofertas')
    axes[0,0].set_xlim(0, 105)
    for bar, val in zip(bars_sk, pct_skills.values):
        axes[0,0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                       f'{val:.1f}%', va='center', fontsize=9)
elif HAS_SKILLS_LONG and df_skills_long is not None:
    top_from_long = df_skills_long['skill'].value_counts().head(15)
    pct_from_long = (top_from_long / df_skills_long['job_id'].nunique() * 100).round(1)
    colors_sk = sns.color_palette('Blues_r', len(pct_from_long))
    bars_sk = axes[0,0].barh(pct_from_long.index, pct_from_long.values, color=colors_sk, edgecolor='white')
    axes[0,0].set_title('14. Skills más frecuentes (job_skills_long)', pad=12)
    axes[0,0].set_xlabel('% de Ofertas')
    for bar, val in zip(bars_sk, pct_from_long.values):
        axes[0,0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                       f'{val:.1f}%', va='center', fontsize=9)
else:
    axes[0,0].text(0.5, 0.5, 'Skills no disponibles', ha='center', va='center',
                   transform=axes[0,0].transAxes, fontsize=12)
    axes[0,0].axis('off')

# G15-16: Rankings usadas vs demandadas (Stack Overflow)
tech_col = 'technology' if 'technology' in df_used.columns else df_used.columns[0]
pct_col  = 'pct' if 'pct' in df_used.columns else ('count' if 'count' in df_used.columns else df_used.columns[1])

top_used   = df_used.sort_values(pct_col, ascending=False).head(12)
top_wanted = df_wanted.sort_values(pct_col, ascending=False).head(12)

axes[0,1].barh(top_used[tech_col], top_used[pct_col],
               color=sns.color_palette('Blues_r', len(top_used)), edgecolor='white')
axes[0,1].set_title('15. Tecnologías Más Usadas (Stack Overflow)', pad=12)
axes[0,1].set_xlabel('% / Count')

axes[1,0].barh(top_wanted[tech_col], top_wanted[pct_col],
               color=sns.color_palette('Greens_r', len(top_wanted)), edgecolor='white')
axes[1,0].set_title('16. Tecnologías Más Demandadas (Stack Overflow)', pad=12)
axes[1,0].set_xlabel('% / Count')

# G17: Brecha Used vs Wanted
common_techs = set(df_used[tech_col]) & set(df_wanted[tech_col])
if common_techs:
    df_gap = pd.merge(
        df_used[df_used[tech_col].isin(common_techs)][[tech_col, pct_col]].rename(columns={pct_col: 'used'}),
        df_wanted[df_wanted[tech_col].isin(common_techs)][[tech_col, pct_col]].rename(columns={pct_col: 'wanted'}),
        on=tech_col
    )
    df_gap['gap'] = df_gap['wanted'] - df_gap['used']
    df_gap = df_gap.sort_values('gap').tail(15)
    colors_gap = [PALETA['success'] if v > 0 else PALETA['warm'] for v in df_gap['gap']]
    axes[1,1].barh(df_gap[tech_col], df_gap['gap'], color=colors_gap, edgecolor='white')
    axes[1,1].axvline(0, color='black', linewidth=0.8)
    axes[1,1].set_title('17. Brecha Wanted − Used (Top 15)', pad=12)
    axes[1,1].set_xlabel('Diferencia (positivo = más demanda que uso)')
else:
    axes[1,1].text(0.5, 0.5, 'Sin tecnologías comunes para calcular brecha',
                   ha='center', va='center', transform=axes[1,1].transAxes, fontsize=11)
    axes[1,1].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '04_stack_tecnologico.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Gráficos 14-17 generados.")


## 📐 Bloque 5 — Análisis Estadístico Avanzado

Regresión salarial, Q-Q plots, pruebas de normalidad y análisis de varianza no paramétrico.


In [ ]:
# ============================================================
# BLOQUE 5: ESTADÍSTICA AVANZADA (Gráficos 18-21)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis Estadístico Avanzado', fontsize=18, fontweight='bold')

# G18: Q-Q Plot de salarios
if sal_col in df_sal.columns and len(df_sal[sal_col].dropna()) > 20:
    salary_sample = df_sal[sal_col].dropna()
    if HAS_STATSMODELS:
        sm.qqplot(salary_sample, line='s', ax=axes[0,0],
                  markerfacecolor=PALETA['accent'], markeredgecolor=PALETA['secondary'],
                  alpha=0.5, markersize=4)
    else:
        (osm, osr), (slope, intercept, r_value) = stats.probplot(salary_sample, dist='norm')
        axes[0,0].scatter(osm, osr, alpha=0.4, color=PALETA['accent'], s=15)
        axes[0,0].plot(osm, slope * np.array(osm) + intercept, color=PALETA['warm'], linewidth=1.5)
        axes[0,0].set_xlabel('Cuantiles teóricos')
        axes[0,0].set_ylabel('Cuantiles muestra')
    axes[0,0].set_title('18. Q-Q Plot: Distribución Salarial', pad=12)
else:
    axes[0,0].text(0.5, 0.5, 'Datos insuficientes', ha='center', va='center',
                   transform=axes[0,0].transAxes, fontsize=12)
    axes[0,0].axis('off')

# G19: Test de Kruskal-Wallis por seniority
sen_col = 'seniority_level' if 'seniority_level' in df_sal.columns else ('seniority' if 'seniority' in df_sal.columns else None)
if sen_col and df_sal[sen_col].notna().any():
    groups = [g[sal_col].dropna() for _, g in df_sal.groupby(sen_col) if len(g) > 5]
    if len(groups) >= 2:
        stat, p_val = kruskal(*groups)
        axes[0,1].axis('off')
        txt = f"Test de Kruskal-Wallis\nSeniority → Salario\n\nH = {stat:.2f}\np = {p_val:.4f}\n\n"
        txt += "✅ Diferencia SIGNIFICATIVA" if p_val < 0.05 else "❌ Sin diferencia significativa"
        axes[0,1].text(0.5, 0.5, txt, ha='center', va='center',
                       transform=axes[0,1].transAxes, fontsize=13,
                       bbox=dict(boxstyle='round', facecolor='#EBF8FF', edgecolor=PALETA['secondary'], linewidth=2))
        axes[0,1].set_title('19. Kruskal-Wallis: Seniority vs Salario', pad=12)
    else:
        axes[0,1].text(0.5, 0.5, 'Grupos insuficientes', ha='center', va='center',
                       transform=axes[0,1].transAxes, fontsize=12)
        axes[0,1].axis('off')
else:
    axes[0,1].axis('off')

# G20: Regresión salario ~ experiencia
exp_col = 'experience_years' if 'experience_years' in df_sal.columns else None
if exp_col and df_sal[[exp_col, sal_col]].notna().all(axis=1).sum() > 20:
    df_reg = df_sal[[exp_col, sal_col]].dropna()
    slope, intercept, r_val, p_val, _ = stats.linregress(df_reg[exp_col], df_reg[sal_col])
    axes[1,0].scatter(df_reg[exp_col], df_reg[sal_col], alpha=0.3,
                      color=PALETA['accent'], s=20)
    x_line = np.linspace(df_reg[exp_col].min(), df_reg[exp_col].max(), 100)
    axes[1,0].plot(x_line, slope * x_line + intercept, color=PALETA['warm'], linewidth=2.5)
    axes[1,0].set_title(f'20. Regresión: Experiencia vs Salario (R²={r_val**2:.3f})', pad=12)
    axes[1,0].set_xlabel('Años de Experiencia')
    axes[1,0].set_ylabel('Salario (€)')
else:
    axes[1,0].text(0.5, 0.5, 'Columna experience_years\nno disponible', ha='center', va='center',
                   transform=axes[1,0].transAxes, fontsize=12)
    axes[1,0].axis('off')

# G21: Distribución de salarios por fuente
if 'source_dataset' in df_sal.columns and df_sal['source_dataset'].notna().any():
    for src, grp in df_sal.groupby('source_dataset'):
        axes[1,1].hist(grp[sal_col].dropna(), bins=25, alpha=0.65, label=src, edgecolor='white')
    axes[1,1].set_title('21. Distribución Salarial por Fuente', pad=12)
    axes[1,1].set_xlabel('Salario Bruto Anual (€)')
    axes[1,1].set_ylabel('Frecuencia')
    axes[1,1].legend()
else:
    axes[1,1].text(0.5, 0.5, 'source_dataset no disponible', ha='center', va='center',
                   transform=axes[1,1].transAxes, fontsize=12)
    axes[1,1].axis('off')

plt.tight_layout()
plt.savefig(IMAGES_DIR / '05_estadistica_avanzada.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Gráficos 18-21 generados.")


## 🌐 Bloque 6 — Visualizaciones Interactivas (Plotly)

Gráficos dinámicos que permiten explorar los datos con zoom, filtros y tooltips.


In [ ]:
# ============================================================
# BLOQUE 6: PLOTLY INTERACTIVO (Gráficos 22-25)
# ============================================================

plotly_template = 'plotly_white'
sen_col = 'seniority_level' if 'seniority_level' in df_sal.columns else ('seniority' if 'seniority' in df_sal.columns else None)
city_col = 'city_clean' if 'city_clean' in df.columns else ('city' if 'city' in df.columns else None)

# G22: Box interactivo — salario por seniority + rol
if sen_col and 'job_title' in df_sal.columns:
    fig22 = px.box(
        df_sal,
        x=sen_col, y=sal_col, color='job_title',
        title='22. Distribución Salarial: Seniority × Rol (interactivo)',
        labels={sal_col: 'Salario (€)', sen_col: ''},
        template=plotly_template,
        color_discrete_sequence=px.colors.qualitative.Bold,
    )
    fig22.update_layout(legend_title_text='Rol')
    fig22.show()
    print("✅ Gráfico 22 (interactivo) mostrado.")

# G23: Scatter salario vs experiencia
exp_col = 'experience_years' if 'experience_years' in df_sal.columns else None
color_col = sen_col if sen_col else 'source_dataset' if 'source_dataset' in df_sal.columns else None
if exp_col:
    fig23 = px.scatter(
        df_sal, x=exp_col, y=sal_col, color=color_col,
        title='23. Experiencia vs Salario',
        labels={sal_col: 'Salario (€)', exp_col: 'Años de experiencia'},
        template=plotly_template,
        opacity=0.65,
    )
    fig23.show()
    print("✅ Gráfico 23 (interactivo) mostrado.")

# G24: Top ciudades por volumen de ofertas
if city_col:
    city_df = df[city_col].value_counts().head(10).reset_index()
    city_df.columns = ['Ciudad', 'Nº Ofertas']
    fig24 = px.bar(city_df, x='Nº Ofertas', y='Ciudad', orientation='h',
                   title='24. Top 10 Ciudades por Volumen de Ofertas',
                   template=plotly_template,
                   color='Nº Ofertas', color_continuous_scale='Blues')
    fig24.update_layout(coloraxis_showscale=False)
    fig24.show()
    print("✅ Gráfico 24 (interactivo) mostrado.")

# G25: Brecha tecnológica (bubble chart)
if common_techs and 'df_gap' in dir():
    fig25 = px.scatter(
        df_gap, x='used', y='wanted', text=tech_col, size=df_gap['gap'].abs() + 1,
        color='gap', color_continuous_scale='RdYlGn',
        title='25. Brecha Tecnológica: Used vs Wanted',
        labels={'used': '% Usan', 'wanted': '% Quieren'},
        template=plotly_template,
    )
    fig25.update_traces(textposition='top center', marker=dict(opacity=0.8))
    max_val = max(df_gap[['used','wanted']].max().max(), 1)
    fig25.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
                    line=dict(color='gray', width=1, dash='dot'))
    fig25.show()
    print("✅ Gráfico 25 (interactivo) mostrado.")


## 🏁 Bloque 7 — Resumen Ejecutivo y Exportación

Panel resumen con los KPIs principales del análisis.


In [ ]:
# ============================================================
# BLOQUE 7: RESUMEN EJECUTIVO — PANEL KPIs
# ============================================================

sal_col = 'salary_clean' if 'salary_clean' in df_sal.columns else 'salary'
city_col = 'city_clean' if 'city_clean' in df.columns else ('city' if 'city' in df.columns else None)
sen_col = 'seniority_level' if 'seniority_level' in df_sal.columns else ('seniority' if 'seniority' in df_sal.columns else None)
modality_col = 'remote_modality' if 'remote_modality' in df.columns else None

kpis = {
    'Total Ofertas': f"{len(df):,}",
    'Con Salario Disponible': f"{len(df_sal):,} ({len(df_sal)/len(df)*100:.1f}%)",
    'Salario Mediano': f"{df_sal[sal_col].median():,.0f} €" if len(df_sal) > 0 else 'N/D',
    'Salario Medio': f"{df_sal[sal_col].mean():,.0f} €" if len(df_sal) > 0 else 'N/D',
    'Top Ciudad': df[city_col].value_counts().index[0] if city_col and df[city_col].notna().any() else 'N/D',
    'Top Modalidad': df[modality_col].value_counts().index[0] if modality_col and df[modality_col].notna().any() else 'N/D',
    'Rol más demandado': df['job_title'].value_counts().index[0] if 'job_title' in df.columns else 'N/D',
    'Top Seniority': df[sen_col].value_counts().index[0] if sen_col and df[sen_col].notna().any() else 'N/D',
}

fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

table_data = [[k, v] for k, v in kpis.items()]
col_widths = [0.4, 0.4]
table = ax.table(cellText=table_data, colLabels=['KPI', 'Valor'],
                 cellLoc='center', loc='center', colWidths=col_widths)
table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 2.2)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor(PALETA['primary'])
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#EBF8FF')
    cell.set_edgecolor('#CBD5E0')

ax.set_title('Panel de KPIs — Mercado Laboral Tech en España',
             fontsize=16, fontweight='bold', pad=20, color=PALETA['primary'])

plt.tight_layout()
plt.savefig(IMAGES_DIR / '07_kpis_resumen.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Panel KPIs generado.")


In [ ]:
# ============================================================
# INVENTARIO DE ARCHIVOS EXPORTADOS
# ============================================================
archivos = sorted(IMAGES_DIR.glob('*.png'))
print(f"\n🖼️  Total de gráficos exportados: {len(archivos)}\n")
for f in archivos:
    size = f.stat().st_size / 1024
    try:
        label = f.relative_to(PROJECT_ROOT)
    except ValueError:
        label = f
    print(f"  ✅ {str(label):<52}  ({size:.0f} KB)")

print("\n🎉 Análisis completado.")
print(f"   Todos los gráficos disponibles en alta calidad (200 DPI) en: {IMAGES_DIR}")


---

## 📋 Conclusiones Analíticas

### Distribución de Vacantes
- Las ciudades con mayor concentración de ofertas tech se identifican en el Gráfico 1.
- El modelo **híbrido** es generalmente la modalidad predominante en el mercado.
- La demanda se concentra en perfiles según seniority (ver Gráfico 3).

### Análisis Salarial
- Existe una diferencia estadísticamente significativa entre los salarios por seniority (Kruskal-Wallis, Gráfico 19).
- La experiencia explica parcialmente la variación salarial (Gráfico 20).

### Stack Tecnológico
- **Python** y **SQL** son las skills con mayor penetración en ofertas de datos.
- El heatmap de co-ocurrencia (Gráfico nuevo) revela qué combinaciones de skills son más habituales.
- Existe una brecha positiva (*wanted > used*) en tecnologías cloud como **AWS** (Gráfico 17).

### Fuente de datos
- Dataset cargado desde: **`{JOBS_SOURCE_PATH.relative_to(PROJECT_ROOT) if JOBS_SOURCE_PATH else 'N/D'}`**
- Datos post-EDA: **{'Sí' if DATOS_POST_EDA else 'No (usando data/clean como fuente)'}**
